In [ ]:
"""
SHWD V1 Benchmark Camera-Ready Controlled SOTA Baselines Suite
======================================================
Account: shwd_v1_archive
Protocol (100% matched to Controlled Ablation Suite):
  - Dataset: SHWD (VOC2028, 7,581 images)
      * train: 5,457 images | val: 607 images | test: 1,517 images (HELD-OUT)
      * classes: 0='hat', 1='person'
  - Models: YOLOv8s, YOLOv9s, YOLOv10s, YOLO12s (all COCO pretrained -> nc=2)
  - Training: 60 epochs, imgsz=640, batch=32, SGD (lr0=0.01, lrf=0.01, momentum=0.937,
    weight_decay=0.0005), cos_lr=True, close_mosaic=10, patience=20, seed=42
  - Evaluation: Held-Out SHWD Test (1,517) + HHW Zero-Shot (Test 1,766 & Full 7,063)
    at imgsz=640, half=False, augment=False (no TTA) + Tesla T4 Latency Benchmark.
"""

import os
import sys
import gc
import time
import shutil
import subprocess
import xml.etree.ElementTree as ET
from pathlib import Path

print("=" * 80, flush=True)
print("[SHWD V1 Benchmark] CONTROLLED SOTA BASELINES SUITE", flush=True)
print("=" * 80, flush=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3.0", "pandas"],
    check=False,
)

os.environ["TENSORBOARD_BINARY"] = ""
os.environ["YOLO_VERBOSE"] = "False"

import pandas as pd
import torch
from ultralytics import YOLO, settings
settings.update({"tensorboard": False})

ngpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
TRAIN_DEVICE = "0,1" if ngpu >= 2 else ("0" if ngpu == 1 else "cpu")
EVAL_DEVICE = "0" if ngpu >= 1 else "cpu"

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

voc_root = None
for p in Path("/kaggle/input").rglob("JPEGImages"):
    if (p.parent / "Annotations").is_dir() and (p.parent / "ImageSets" / "Main" / "test.txt").exists():
        voc_root = p.parent
        break
assert voc_root is not None, "VOC2028 not found!"

SHWD_DIR = WORK_DIR / "SHWD_YOLO"
for sp in ("train", "val", "test"):
    (SHWD_DIR / "images" / sp).mkdir(parents=True, exist_ok=True)
    (SHWD_DIR / "labels" / sp).mkdir(parents=True, exist_ok=True)

SHWD_MAP = {"hat": 0, "helmet": 0, "person": 1, "head": 1}

def build_voc_yolo(root_dir: Path, id_file: Path, out_dir: Path, split_name: str, cls_map: dict):
    ids = [x.strip() for x in id_file.read_text(encoding="utf-8").splitlines() if x.strip()]
    n_ok = 0
    for stem in ids:
        xp = root_dir / "Annotations" / f"{stem}.xml"
        if not xp.exists():
            continue
        img_p = None
        for ext in (".jpg", ".JPG", ".png", ".jpeg"):
            cand = root_dir / "JPEGImages" / f"{stem}{ext}"
            if cand.exists():
                img_p = cand
                break
        if img_p is None:
            continue
        root = ET.parse(xp).getroot()
        sz = root.find("size")
        iw, ih = int(sz.find("width").text), int(sz.find("height").text)
        if iw <= 0 or ih <= 0:
            continue
        lines = []
        for obj in root.findall("object"):
            cname = obj.find("name").text.strip().lower()
            if cname not in cls_map:
                continue
            cid = cls_map[cname]
            bb = obj.find("bndbox")
            xmin = max(0.0, min(float(bb.find("xmin").text), float(iw)))
            ymin = max(0.0, min(float(bb.find("ymin").text), float(ih)))
            xmax = max(0.0, min(float(bb.find("xmax").text), float(iw)))
            ymax = max(0.0, min(float(bb.find("ymax").text), float(ih)))
            if xmax <= xmin or ymax <= ymin:
                continue
            cx = ((xmin + xmax) / 2.0) / iw
            cy = ((ymin + ymax) / 2.0) / ih
            bw = (xmax - xmin) / iw
            bh = (ymax - ymin) / ih
            lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        dst_img = out_dir / "images" / split_name / img_p.name
        if not dst_img.exists():
            try:
                os.symlink(img_p, dst_img)
            except Exception:
                shutil.copy2(img_p, dst_img)
        (out_dir / "labels" / split_name / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
        n_ok += 1
    print(f"  SHWD {split_name}: {n_ok} images", flush=True)

for sp in ("train", "val", "test"):
    build_voc_yolo(voc_root, voc_root / "ImageSets" / "Main" / f"{sp}.txt", SHWD_DIR, sp, SHWD_MAP)

shwd_yaml = SHWD_DIR / "shwd.yaml"
shwd_yaml.write_text(
    f"path: {SHWD_DIR.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 2\nnames: ['hat', 'person']\n",
    encoding="utf-8",
)

hhw_root = None
for p in Path("/kaggle/input").rglob("Annotation"):
    if (p.parent / "JPEGImage").is_dir() and p.parent.name in ("Test", "Train"):
        hhw_root = p.parent.parent
        break

hhw_configs = {}
if hhw_root is not None:
    for mode, cmap, names_list in [
        ("hhw_2class", {"helmet": 0, "head": 1}, ["hat", "person"]),
        ("hhw_hatonly", {"helmet": 0}, ["hat"]),
    ]:
        for subset_name, subdirs in [("test", ["Test"]), ("full", ["Train", "Test"])]:
            out_hhw = WORK_DIR / f"HHW_{mode}_{subset_name}"
            (out_hhw / "images" / "val").mkdir(parents=True, exist_ok=True)
            (out_hhw / "labels" / "val").mkdir(parents=True, exist_ok=True)
            n_img = 0
            for sd in subdirs:
                ad, jd = hhw_root / sd / "Annotation", hhw_root / sd / "JPEGImage"
                if not ad.exists():
                    continue
                for xp in sorted(ad.glob("*.xml")):
                    stem = xp.stem
                    img_p = next((jd / f"{stem}{ext}" for ext in (".jpg", ".JPG", ".png", ".jpeg") if (jd / f"{stem}{ext}").exists()), None)
                    if img_p is None:
                        continue
                    root = ET.parse(xp).getroot()
                    sz = root.find("size")
                    iw, ih = int(sz.find("width").text), int(sz.find("height").text)
                    lines = []
                    for obj in root.findall("object"):
                        cname = obj.find("name").text.strip().lower()
                        if cname not in cmap:
                            continue
                        cid = cmap[cname]
                        bb = obj.find("bndbox")
                        xmin = max(0.0, min(float(bb.find("xmin").text), float(iw)))
                        ymin = max(0.0, min(float(bb.find("ymin").text), float(ih)))
                        xmax = max(0.0, min(float(bb.find("xmax").text), float(iw)))
                        ymax = max(0.0, min(float(bb.find("ymax").text), float(ih)))
                        if xmax <= xmin or ymax <= ymin:
                            continue
                        cx = ((xmin + xmax) / 2.0) / iw
                        cy = ((ymin + ymax) / 2.0) / ih
                        bw = (xmax - xmin) / iw
                        bh = (ymax - ymin) / ih
                        lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
                    dst_img = out_hhw / "images" / "val" / f"{sd}_{img_p.name}"
                    if not dst_img.exists():
                        try:
                            os.symlink(img_p, dst_img)
                        except Exception:
                            shutil.copy2(img_p, dst_img)
                    (out_hhw / "labels" / "val" / f"{sd}_{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
                    n_img += 1
            yp = out_hhw / "hhw.yaml"
            yp.write_text(
                f"path: {out_hhw.resolve()}\ntrain: images/val\nval: images/val\nnc: {len(names_list)}\nnames: {names_list}\n",
                encoding="utf-8",
            )
            hhw_configs[f"{mode}_{subset_name}"] = yp

def extract_metrics(val_res):
    res = {
        "mAP50": round(float(val_res.box.map50) * 100.0, 2),
        "mAP50_95": round(float(val_res.box.map) * 100.0, 2),
        "precision": round(float(val_res.box.mp) * 100.0, 2),
        "recall": round(float(val_res.box.mr) * 100.0, 2),
    }
    try:
        p0, r0, ap50_0, ap_0 = val_res.box.class_result(0)
        res.update({"hat_P": round(float(p0)*100, 2), "hat_R": round(float(r0)*100, 2), "hat_AP50": round(float(ap50_0)*100, 2), "hat_AP50_95": round(float(ap_0)*100, 2)})
    except Exception:
        pass
    try:
        p1, r1, ap50_1, ap_1 = val_res.box.class_result(1)
        res.update({"person_P": round(float(p1)*100, 2), "person_R": round(float(r1)*100, 2), "person_AP50": round(float(ap50_1)*100, 2), "person_AP50_95": round(float(ap_1)*100, 2)})
    except Exception:
        pass
    return res

OUT_DIR = WORK_DIR / "SHWD V1 Benchmark_Baselines"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = [
    ("YOLOv8s", "yolov8s.pt"),
    ("YOLOv9s", "yolov9s.pt"),
    ("YOLOv10s", "yolov10s.pt"),
    ("YOLO12s", "yolo12s.pt"),
]

SEED = 42
EPOCHS = 60
BATCH_SIZE = 32
IMGSZ = 640
PATIENCE = 20

records, hhw_records = [], []

for label, weights in MODELS:
    print(f"\n{'='*80}\n[BASELINE TRAIN] {label} ({weights})\n{'='*80}", flush=True)
    model = YOLO(weights)
    t0 = time.time()
    model.train(
        data=str(shwd_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        device=TRAIN_DEVICE,
        seed=SEED,
        deterministic=True,
        optimizer="SGD",
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        cos_lr=True,
        patience=PATIENCE,
        close_mosaic=10,
        cache=True,
        save=True,
        val=True,
        plots=False,
        project=str(OUT_DIR / "runs"),
        name=label,
        exist_ok=True,
        verbose=False,
    )
    train_min = (time.time() - t0) / 60.0
    best_pt = Path(model.trainer.save_dir) / "weights" / "best.pt"
    eval_m = YOLO(str(best_pt))
    params = sum(p.numel() for p in eval_m.model.parameters())

    test_res = eval_m.val(
        data=str(shwd_yaml), split="test", imgsz=IMGSZ, batch=16,
        device=EVAL_DEVICE, half=False, augment=False, verbose=False,
    )
    m_dict = extract_metrics(test_res)

    # Latency T4 PyTorch FP32/FP16
    m_gpu = eval_m.model.cuda().eval()
    x32 = torch.randn(1, 3, 640, 640, device="cuda", dtype=torch.float32)
    with torch.no_grad():
        for _ in range(50): _ = m_gpu(x32)
        torch.cuda.synchronize()
        ts = time.perf_counter()
        for _ in range(200): _ = m_gpu(x32)
        torch.cuda.synchronize()
        fp32_ms = (time.perf_counter() - ts) / 200.0 * 1000.0
    m_gpu.half()
    x16 = x32.half()
    with torch.no_grad():
        for _ in range(50): _ = m_gpu(x16)
        torch.cuda.synchronize()
        ts = time.perf_counter()
        for _ in range(200): _ = m_gpu(x16)
        torch.cuda.synchronize()
        fp16_ms = (time.perf_counter() - ts) / 200.0 * 1000.0

    rec = {
        "model": label, "params": params, "train_time_min": round(train_min, 2),
        "pyt_fp32_ms": round(fp32_ms, 2), "pyt_fp32_fps": round(1000.0/fp32_ms, 1),
        "pyt_fp16_ms": round(fp16_ms, 2), "pyt_fp16_fps": round(1000.0/fp16_ms, 1),
        **m_dict,
    }
    records.append(rec)
    pd.DataFrame(records).to_csv(OUT_DIR / "shwd_controlled_baselines.csv", index=False)
    print(f"[TEST {label}] {rec}", flush=True)

    for hname, hyp in hhw_configs.items():
        if "hatonly" in hname:
            hr = eval_m.val(data=str(hyp), split="val", imgsz=IMGSZ, batch=16, device=EVAL_DEVICE, half=False, augment=False, classes=[0], verbose=False)
        else:
            hr = eval_m.val(data=str(hyp), split="val", imgsz=IMGSZ, batch=16, device=EVAL_DEVICE, half=False, augment=False, verbose=False)
        hm = extract_metrics(hr)
        hhw_records.append({"model": label, "hhw_eval": hname, **hm})
        pd.DataFrame(hhw_records).to_csv(OUT_DIR / "hhw_zeroshot_baselines.csv", index=False)

    del model, eval_m, m_gpu
    torch.cuda.empty_cache()
    gc.collect()

print("\n[SUMMARY] Controlled SOTA Baselines:", flush=True)
print(pd.DataFrame(records).to_string(index=False), flush=True)
